# Gemma4-GraphFix: Autonomous SWE Developer Agent (Kaggle Competition Edition)

**Competition:** [Google - The Gemma 4 Developer Agent Competition](https://www.kaggle.com/competitions/gemma-4-developer-agent)  
**Critical Deadlines:**
- **Rules Acceptance Deadline:** `2026-11-25` (Mandatory before final submission)
- **Final Submission Deadline:** `2026-12-02 23:59 UTC`

## 5-Stage Architecture Optimized for Gemma 4 (`31B-QAT` & `E4B-QAT`)
1. **AST Fault Localization:** Python `ast` symbol/call-graph indexing + stacktrace innermost-frame weighting + BM25 sub-token IDF with 2-hop call-graph propagation (separating production files from read-only tests).
2. **Minimal Context Slicing:** Token-budgeted line-numbered (`L0001:`) AST window with imports, class headers, and 1-hop caller/callee signatures to fit cleanly inside 32K/128K Gemma 4 budgets.
3. **Multi-Candidate Patch Synthesis:** Temperature & strategy-stratified candidates (`surgical_greedy`, `boundary_guard`, `state_invariant`) supporting unified diffs and `SEARCH/REPLACE` blocks after stripping `<|channel>thought` reasoning channels.
4. **Test-Execution Self-Refinement & Regression Guard:** Isolated sandbox evaluation with AST syntax pre-check, strict anti-test-tampering policy (`allow_test_edits=False`), and compact traceback-delta refinement loops.
5. **Deterministic Patch Ranking:** Lexicographic scoring (`policy_ok > ast_valid > target_pass > zero_regression > pass_ratio > loc_alignment > -blast_radius > sha256_tiebreak`).


In [ ]:
import json
import os
import platform
import sys
from pathlib import Path

WORK_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
PKG_ROOT = WORK_ROOT / "src"
PKG_ROOT.mkdir(parents=True, exist_ok=True)
if str(PKG_ROOT) not in sys.path:
    sys.path.insert(0, str(PKG_ROOT))

hw_info = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "work_root": str(WORK_ROOT),
    "kaggle_input_exists": Path("/kaggle/input").exists(),
}
try:
    import torch
    hw_info["cuda_available"] = bool(torch.cuda.is_available())
    hw_info["gpu_count"] = int(torch.cuda.device_count()) if torch.cuda.is_available() else 0
    hw_info["gpus"] = [torch.cuda.get_device_name(i) for i in range(hw_info["gpu_count"])]
except Exception:
    hw_info["cuda_available"] = False
    hw_info["gpu_count"] = 0
    hw_info["gpus"] = []

print("Environment:", json.dumps(hw_info, indent=2))
if Path("/kaggle/input").exists():
    print("Mounted /kaggle/input:", [p.name for p in sorted(Path("/kaggle/input").iterdir())])

In [ ]:
EMBEDDED_PKG_FILES = {'__init__.py': '"""Gemma4-GraphFix: Autonomous SWE Bug-Fixing Agent optimized for Gemma 4."""\n\nfrom .ast_localizer import (\n    ASTFaultLocalizer,\n    CodeEntity,\n    LocalizationResult,\n    SuspiciousEntity,\n)\nfrom .context_slicer import ContextSlice, MinimalContextSlicer\nfrom .diff_engine import (\n    BlastRadius,\n    DiffApplyResult,\n    FileDiff,\n    Hunk,\n    apply_unified_diff,\n    compute_blast_radius,\n    parse_search_replace_blocks,\n    parse_unified_diff,\n    search_replace_to_unified_diff,\n    validate_python_syntax,\n)\nfrom .gemma4_runtime import (\n    DeterministicRuleRuntime,\n    Gemma4TransformersRuntime,\n    Gemma4VLLMRuntime,\n    extract_diff_payload,\n    strip_gemma4_thoughts,\n)\nfrom .patch_pipeline import (\n    CandidateEvaluation,\n    CandidatePatch,\n    DeterministicPatchRanker,\n    RegressionGuard,\n    TestExecutionReport,\n)\nfrom .agent_orchestrator import AgentRunResult, AutonomousSWEAgent\n\n__all__ = [\n    "ASTFaultLocalizer",\n    "AgentRunResult",\n    "AutonomousSWEAgent",\n    "BlastRadius",\n    "CandidateEvaluation",\n    "CandidatePatch",\n    "CodeEntity",\n    "ContextSlice",\n    "DeterministicPatchRanker",\n    "DeterministicRuleRuntime",\n    "DiffApplyResult",\n    "FileDiff",\n    "Gemma4TransformersRuntime",\n    "Gemma4VLLMRuntime",\n    "Hunk",\n    "LocalizationResult",\n    "MinimalContextSlicer",\n    "RegressionGuard",\n    "SuspiciousEntity",\n    "TestExecutionReport",\n    "apply_unified_diff",\n    "compute_blast_radius",\n    "extract_diff_payload",\n    "parse_search_replace_blocks",\n    "parse_unified_diff",\n    "search_replace_to_unified_diff",\n    "strip_gemma4_thoughts",\n    "validate_python_syntax",\n]\n', 'ast_localizer.py': '"""AST Fault Localization with symbol/call-graph propagation for Gemma 4 SWE agents."""\n\nfrom __future__ import annotations\n\nimport ast\nimport math\nimport os\nimport re\nfrom collections import defaultdict\nfrom dataclasses import dataclass, field\nfrom pathlib import Path\nfrom typing import Iterable, Sequence\n\nIGNORED_DIRS = frozenset(\n    {\n        ".git",\n        ".hg",\n        ".svn",\n        "__pycache__",\n        ".pytest_cache",\n        ".mypy_cache",\n        ".venv",\n        "venv",\n        "env",\n        "node_modules",\n        "build",\n        "dist",\n        ".eggs",\n    }\n)\n\nSTOPWORDS = frozenset(\n    {\n        "the",\n        "and",\n        "for",\n        "with",\n        "that",\n        "this",\n        "from",\n        "into",\n        "when",\n        "where",\n        "what",\n        "which",\n        "while",\n        "should",\n        "would",\n        "could",\n        "fail",\n        "fails",\n        "failed",\n        "error",\n        "exception",\n        "traceback",\n        "line",\n        "file",\n        "none",\n        "true",\n        "false",\n        "self",\n        "cls",\n        "args",\n        "kwargs",\n        "return",\n        "returns",\n        "raise",\n        "raises",\n        "test",\n        "tests",\n        "def",\n        "class",\n        "import",\n        "bug",\n        "issue",\n        "fix",\n    }\n)\n\nTRACEBACK_RE = re.compile(\n    r\'File\\s+["\\\'](?P<path>[^"\\\']+\\.py)["\\\']\\s*,\\s*line\\s+(?P<line>\\d+)(?:\\s*,\\s*in\\s+(?P<func>[A-Za-z_][A-Za-z0-9_<>]*))?\'\n)\nPYTEST_LINE_RE = re.compile(r"(?P<path>[A-Za-z0-9_./\\\\-]+\\.py):(?P<line>\\d+):")\nFILE_MENTION_RE = re.compile(r"(?P<path>[A-Za-z0-9_./\\\\-]+\\.py)")\n\n\ndef split_identifier_tokens(text: str) -> list[str]:\n    """Split snake_case and CamelCase identifiers into lowercase search tokens."""\n    raw_words = re.findall(r"[A-Za-z_][A-Za-z0-9_]*", text)\n    tokens: list[str] = []\n    for word in raw_words:\n        for snake_part in word.split("_"):\n            if not snake_part:\n                continue\n            camel_parts = re.findall(\n                r"[A-Z]+(?=[A-Z][a-z0-9]|$)|[A-Z]?[a-z]+|[0-9]+", snake_part\n            )\n            if camel_parts:\n                for cp in camel_parts:\n                    low = cp.lower()\n                    if len(low) >= 2 and low not in STOPWORDS:\n                        tokens.append(low)\n            else:\n                low = snake_part.lower()\n                if len(low) >= 2 and low not in STOPWORDS:\n                    tokens.append(low)\n    return tokens\n\n\ndef is_test_path(rel_path: str) -> bool:\n    """Return True when a repository-relative path represents a test file."""\n    norm = rel_path.replace("\\\\", "/").lower()\n    parts = norm.split("/")\n    filename = parts[-1]\n    if any(p in ("tests", "test", "testing") for p in parts[:-1]):\n        return True\n    return filename.startswith("test_") or filename.endswith("_test.py")\n\n\n@dataclass(frozen=True)\nclass TracebackHit:\n    """Single parsed stack frame from a traceback or pytest output."""\n\n    rel_path: str\n    line: int\n    func_name: str\n    depth_weight: float\n\n\n@dataclass\nclass CodeEntity:\n    """AST-indexed function, method, or class inside a Python repository."""\n\n    entity_id: str\n    rel_path: str\n    qualname: str\n    name: str\n    kind: str  # "function", "async_function", "method", "class"\n    start_line: int\n    end_line: int\n    parent_class: str | None = None\n    signature: str = ""\n    docstring: str = ""\n    calls: set[str] = field(default_factory=set)\n    identifiers: set[str] = field(default_factory=set)\n    imports: set[str] = field(default_factory=set)\n    is_test: bool = False\n\n    def contains_line(self, line: int) -> bool:\n        return self.start_line <= line <= self.end_line\n\n    @property\n    def span_lines(self) -> int:\n        return max(1, self.end_line - self.start_line + 1)\n\n\n@dataclass\nclass SuspiciousEntity:\n    """Ranked candidate fault location with decomposed evidence."""\n\n    entity: CodeEntity\n    score: float\n    direct_score: float\n    graph_score: float\n    reasons: list[str] = field(default_factory=list)\n\n\n@dataclass\nclass LocalizationResult:\n    """Complete AST fault localization output."""\n\n    ranked_entities: list[SuspiciousEntity]\n    ranked_files: list[tuple[str, float]]\n    linked_tests: list[str]\n    traceback_hits: list[TracebackHit]\n    call_graph_edges: dict[str, list[str]]\n    total_entities_indexed: int\n    total_files_indexed: int\n\n\nclass _EntityVisitor(ast.NodeVisitor):\n    """Collect AST symbol definitions, signatures, calls, and lexical tokens."""\n\n    def __init__(\n        self, rel_path: str, module_name: str, source_lines: list[str]\n    ) -> None:\n        self.rel_path = rel_path\n        self.module_name = module_name\n        self.source_lines = source_lines\n        self.is_test = is_test_path(rel_path)\n        self.module_imports: set[str] = set()\n        self.entities: list[CodeEntity] = []\n        self._class_stack: list[str] = []\n        self._entity_stack: list[CodeEntity] = []\n\n    def visit_Import(self, node: ast.Import) -> None:\n        for alias in node.names:\n            self.module_imports.add(alias.name.split(".")[0])\n            if self._entity_stack:\n                self._entity_stack[-1].imports.add(alias.name)\n        self.generic_visit(node)\n\n    def visit_ImportFrom(self, node: ast.ImportFrom) -> None:\n        mod = node.module or ""\n        if mod:\n            self.module_imports.add(mod.split(".")[0])\n        for alias in node.names:\n            self.module_imports.add(alias.name)\n            if self._entity_stack:\n                self._entity_stack[-1].imports.add(f"{mod}.{alias.name}".strip("."))\n        self.generic_visit(node)\n\n    def visit_ClassDef(self, node: ast.ClassDef) -> None:\n        end_line = getattr(node, "end_lineno", node.lineno) or node.lineno\n        qual_parts = [self.module_name, *self._class_stack, node.name]\n        qualname = ".".join(p for p in qual_parts if p)\n        bases = [ast.unparse(b) for b in node.bases] if hasattr(ast, "unparse") else []\n        sig = (\n            f"class {node.name}({\', \'.join(bases)}):"\n            if bases\n            else f"class {node.name}:"\n        )\n        doc = (\n            (ast.get_docstring(node) or "").splitlines()[0]\n            if ast.get_docstring(node)\n            else ""\n        )\n        entity = CodeEntity(\n            entity_id=f"{self.rel_path}::{qualname}",\n            rel_path=self.rel_path,\n            qualname=qualname,\n            name=node.name,\n            kind="class",\n            start_line=node.lineno,\n            end_line=end_line,\n            parent_class=self._class_stack[-1] if self._class_stack else None,\n            signature=sig,\n            docstring=doc,\n            imports=set(self.module_imports),\n            is_test=self.is_test,\n        )\n        entity.identifiers.update(split_identifier_tokens(node.name))\n        if doc:\n            entity.identifiers.update(split_identifier_tokens(doc))\n        self.entities.append(entity)\n\n        self._class_stack.append(node.name)\n        self._entity_stack.append(entity)\n        self.generic_visit(node)\n        self._entity_stack.pop()\n        self._class_stack.pop()\n\n    def _visit_func(\n        self, node: ast.FunctionDef | ast.AsyncFunctionDef, is_async: bool\n    ) -> None:\n        end_line = getattr(node, "end_lineno", node.lineno) or node.lineno\n        parent_cls = self._class_stack[-1] if self._class_stack else None\n        qual_parts = [self.module_name, *self._class_stack, node.name]\n        qualname = ".".join(p for p in qual_parts if p)\n        prefix = "async def" if is_async else "def"\n        arg_names = [a.arg for a in node.args.posonlyargs + node.args.args]\n        if node.args.vararg:\n            arg_names.append(f"*{node.args.vararg.arg}")\n        arg_names.extend(a.arg for a in node.args.kwonlyargs)\n        if node.args.kwarg:\n            arg_names.append(f"**{node.args.kwarg.arg}")\n        ret_ann = ""\n        if node.returns is not None and hasattr(ast, "unparse"):\n            try:\n                ret_ann = f" -> {ast.unparse(node.returns)}"\n            except Exception:\n                ret_ann = ""\n        sig = f"{prefix} {node.name}({\', \'.join(arg_names)}){ret_ann}:"\n        raw_doc = ast.get_docstring(node) or ""\n        first_doc = raw_doc.splitlines()[0] if raw_doc else ""\n        kind = (\n            "method" if parent_cls else ("async_function" if is_async else "function")\n        )\n\n        entity = CodeEntity(\n            entity_id=f"{self.rel_path}::{qualname}",\n            rel_path=self.rel_path,\n            qualname=qualname,\n            name=node.name,\n            kind=kind,\n            start_line=node.lineno,\n            end_line=end_line,\n            parent_class=parent_cls,\n            signature=sig,\n            docstring=first_doc,\n            imports=set(self.module_imports),\n            is_test=self.is_test,\n        )\n        body_slice = "\\n".join(self.source_lines[node.lineno - 1 : end_line])\n        entity.identifiers.update(split_identifier_tokens(node.name))\n        entity.identifiers.update(split_identifier_tokens(body_slice))\n        if parent_cls:\n            entity.identifiers.update(split_identifier_tokens(parent_cls))\n\n        self.entities.append(entity)\n        self._entity_stack.append(entity)\n        self.generic_visit(node)\n        self._entity_stack.pop()\n\n    def visit_FunctionDef(self, node: ast.FunctionDef) -> None:\n        self._visit_func(node, is_async=False)\n\n    def visit_AsyncFunctionDef(self, node: ast.AsyncFunctionDef) -> None:\n        self._visit_func(node, is_async=True)\n\n    def visit_Call(self, node: ast.Call) -> None:\n        if self._entity_stack:\n            target_name = None\n            if isinstance(node.func, ast.Name):\n                target_name = node.func.id\n            elif isinstance(node.func, ast.Attribute):\n                target_name = node.func.attr\n            if target_name:\n                self._entity_stack[-1].calls.add(target_name)\n        self.generic_visit(node)\n\n\nclass ASTFaultLocalizer:\n    """Hybrid AST symbol-graph + traceback + BM25 fault localizer for Python repos."""\n\n    def __init__(self, repo_root: str | Path) -> None:\n        self.repo_root = Path(repo_root).resolve()\n        self.entities: dict[str, CodeEntity] = {}\n        self.file_entities: dict[str, list[CodeEntity]] = defaultdict(list)\n        self.name_to_ids: dict[str, list[str]] = defaultdict(list)\n        self.call_graph: dict[str, list[str]] = defaultdict(list)\n        self.reverse_graph: dict[str, list[str]] = defaultdict(list)\n        self.doc_freq: dict[str, int] = defaultdict(int)\n        self.file_count = 0\n        self._indexed = False\n\n    def index_repository(self) -> None:\n        """Walk repository Python files and construct AST symbol & call graphs."""\n        self.entities.clear()\n        self.file_entities.clear()\n        self.name_to_ids.clear()\n        self.call_graph.clear()\n        self.reverse_graph.clear()\n        self.doc_freq.clear()\n        self.file_count = 0\n\n        for root, dirs, files in os.walk(self.repo_root):\n            dirs[:] = sorted(\n                d for d in dirs if d not in IGNORED_DIRS and not d.startswith(".")\n            )\n            for fname in sorted(files):\n                if not fname.endswith(".py"):\n                    continue\n                full_path = Path(root) / fname\n                rel_path = full_path.relative_to(self.repo_root).as_posix()\n                try:\n                    text = full_path.read_text(encoding="utf-8", errors="replace")\n                    tree = ast.parse(text, filename=rel_path)\n                except SyntaxError:\n                    continue\n                self.file_count += 1\n                mod_name = rel_path[:-3].replace("/", ".")\n                if mod_name.endswith(".__init__"):\n                    mod_name = mod_name[: -len(".__init__")]\n                visitor = _EntityVisitor(rel_path, mod_name, text.splitlines())\n                visitor.visit(tree)\n                for entity in visitor.entities:\n                    self.entities[entity.entity_id] = entity\n                    self.file_entities[rel_path].append(entity)\n                    self.name_to_ids[entity.name].append(entity.entity_id)\n                    for tok in entity.identifiers:\n                        self.doc_freq[tok] += 1\n\n        # Resolve symbol calls into directed edges (caller -> callee)\n        for caller_id, caller in self.entities.items():\n            seen_targets: set[str] = set()\n            for call_name in caller.calls:\n                for target_id in self.name_to_ids.get(call_name, ()):\n                    if target_id == caller_id or target_id in seen_targets:\n                        continue\n                    target = self.entities[target_id]\n                    if target.is_test and not caller.is_test:\n                        continue\n                    seen_targets.add(target_id)\n                    self.call_graph[caller_id].append(target_id)\n                    self.reverse_graph[target_id].append(caller_id)\n\n        self._indexed = True\n\n    def _normalize_repo_path(self, raw_path: str) -> str | None:\n        cleaned = raw_path.replace("\\\\", "/").lstrip("./")\n        if cleaned in self.file_entities:\n            return cleaned\n        # Handle absolute or nested paths by longest suffix match\n        candidates = [\n            rel\n            for rel in self.file_entities\n            if cleaned.endswith("/" + rel)\n            or rel.endswith("/" + cleaned)\n            or rel == cleaned\n        ]\n        if candidates:\n            return max(candidates, key=len)\n        base = Path(cleaned).name\n        by_name = [rel for rel in self.file_entities if Path(rel).name == base]\n        return by_name[0] if len(by_name) == 1 else None\n\n    def parse_traceback_hits(self, issue_or_trace: str) -> list[TracebackHit]:\n        """Extract stack frames and pytest location hits with innermost boost."""\n        raw_hits: list[tuple[str, int, str]] = []\n        for match in TRACEBACK_RE.finditer(issue_or_trace):\n            rel = self._normalize_repo_path(match.group("path"))\n            if rel:\n                raw_hits.append(\n                    (rel, int(match.group("line")), match.group("func") or "")\n                )\n        for match in PYTEST_LINE_RE.finditer(issue_or_trace):\n            rel = self._normalize_repo_path(match.group("path"))\n            if rel:\n                raw_hits.append((rel, int(match.group("line")), ""))\n\n        if not raw_hits:\n            return []\n\n        # Deduplicate while preserving order; deepest stack frames get highest weight\n        deduped: list[tuple[str, int, str]] = []\n        seen: set[tuple[str, int]] = set()\n        for item in raw_hits:\n            key = (item[0], item[1])\n            if key not in seen:\n                seen.add(key)\n                deduped.append(item)\n\n        hits: list[TracebackHit] = []\n        total = len(deduped)\n        for idx, (rel, line, func_name) in enumerate(deduped):\n            # Innermost frame (largest idx) receives up to 1.6x boost\n            depth_weight = 1.0 + (0.6 * (idx / max(1, total - 1))) if total > 1 else 1.4\n            hits.append(\n                TracebackHit(\n                    rel_path=rel,\n                    line=line,\n                    func_name=func_name,\n                    depth_weight=round(depth_weight, 4),\n                )\n            )\n        return hits\n\n    def _idf(self, token: str) -> float:\n        n = max(1, len(self.entities))\n        df = self.doc_freq.get(token, 0)\n        return math.log(1.0 + (n - df + 0.5) / (df + 0.5))\n\n    def localize(\n        self,\n        issue_text: str,\n        traceback_text: str = "",\n        top_k: int = 8,\n        include_tests: bool = False,\n    ) -> LocalizationResult:\n        """Rank suspicious AST entities and files using hybrid static + traceback evidence."""\n        if not self._indexed:\n            self.index_repository()\n\n        combined_query = f"{issue_text}\\n{traceback_text}".strip()\n        tb_hits = self.parse_traceback_hits(combined_query)\n        mentioned_files: set[str] = set()\n        for match in FILE_MENTION_RE.finditer(combined_query):\n            norm = self._normalize_repo_path(match.group("path"))\n            if norm:\n                mentioned_files.add(norm)\n\n        exact_identifiers = set(\n            re.findall(r"\\b[A-Za-z_][A-Za-z0-9_]{2,}\\b", combined_query)\n        )\n        query_tokens = set(split_identifier_tokens(combined_query))\n\n        direct_scores: dict[str, float] = defaultdict(float)\n        reasons_map: dict[str, list[str]] = defaultdict(list)\n\n        # 1. Exact stacktrace / failing test line alignment\n        for hit in tb_hits:\n            for ent in self.file_entities.get(hit.rel_path, ()):\n                if ent.contains_line(hit.line):\n                    specificity = 1.0 if ent.kind != "class" else 0.45\n                    w = 3.5 * hit.depth_weight * specificity\n                    direct_scores[ent.entity_id] += w\n                    reasons_map[ent.entity_id].append(\n                        f"traceback_line:{hit.rel_path}:{hit.line}(+{w:.2f})"\n                    )\n                elif hit.func_name and ent.name == hit.func_name:\n                    w = 2.4 * hit.depth_weight\n                    direct_scores[ent.entity_id] += w\n                    reasons_map[ent.entity_id].append(\n                        f"traceback_func:{hit.func_name}(+{w:.2f})"\n                    )\n\n        # 2. Mentioned files + exact symbol hits + IDF-weighted subtoken overlap\n        for eid, ent in self.entities.items():\n            if ent.rel_path in mentioned_files:\n                w = 1.6 if not ent.is_test else 0.4\n                direct_scores[eid] += w\n                reasons_map[eid].append(f"file_mention:{ent.rel_path}(+{w:.2f})")\n\n            if ent.name in exact_identifiers and ent.name not in STOPWORDS:\n                w = 2.8 if ent.kind != "class" else 1.8\n                direct_scores[eid] += w\n                reasons_map[eid].append(f"exact_symbol:{ent.name}(+{w:.2f})")\n\n            overlap = query_tokens & ent.identifiers\n            if overlap:\n                idf_sum = sum(self._idf(tok) for tok in overlap)\n                # Normalize slightly by entity size to favor crisp fault functions\n                len_norm = 1.0 + 0.15 * math.log1p(ent.span_lines)\n                bm25_like = (idf_sum / len_norm) * 0.75\n                if bm25_like > 0.05:\n                    direct_scores[eid] += bm25_like\n                    top_toks = ",".join(sorted(overlap)[:4])\n                    reasons_map[eid].append(f"token_idf[{top_toks}](+{bm25_like:.2f})")\n\n        # 3. Call-graph 2-hop propagation (both test->callee and symptom->callee/caller)\n        graph_scores: dict[str, float] = defaultdict(float)\n        # First hop from all directly scored entities (including test entities calling production!)\n        for src_id, src_score in list(direct_scores.items()):\n            if src_score <= 0.1:\n                continue\n            for callee_id in self.call_graph.get(src_id, ()):\n                prop = src_score * 0.48\n                graph_scores[callee_id] += prop\n                reasons_map[callee_id].append(\n                    f"called_by:{self.entities[src_id].name}(+{prop:.2f})"\n                )\n                for hop2_id in self.call_graph.get(callee_id, ()):\n                    prop2 = src_score * 0.20\n                    graph_scores[hop2_id] += prop2\n                    reasons_map[hop2_id].append(\n                        f"2hop_callee_via:{self.entities[callee_id].name}(+{prop2:.2f})"\n                    )\n            for caller_id in self.reverse_graph.get(src_id, ()):\n                if not self.entities[caller_id].is_test:\n                    prop_up = src_score * 0.22\n                    graph_scores[caller_id] += prop_up\n                    reasons_map[caller_id].append(\n                        f"calls:{self.entities[src_id].name}(+{prop_up:.2f})"\n                    )\n\n        # Combine and rank deterministically\n        ranked: list[SuspiciousEntity] = []\n        file_scores: dict[str, float] = defaultdict(float)\n        linked_tests: list[tuple[float, str]] = []\n\n        for eid, ent in self.entities.items():\n            d_score = direct_scores.get(eid, 0.0)\n            g_score = graph_scores.get(eid, 0.0)\n            total = d_score + g_score\n            if total <= 0.0:\n                continue\n            if ent.is_test:\n                linked_tests.append((total, ent.rel_path))\n                if not include_tests:\n                    continue\n            # Slight preference for leaf methods/functions over whole class wrapper\n            if ent.kind == "class":\n                total *= 0.72\n            ranked.append(\n                SuspiciousEntity(\n                    entity=ent,\n                    score=round(total, 6),\n                    direct_score=round(d_score, 6),\n                    graph_score=round(g_score, 6),\n                    reasons=reasons_map.get(eid, [])[:6],\n                )\n            )\n            file_scores[ent.rel_path] = max(file_scores[ent.rel_path], round(total, 6))\n\n        ranked.sort(\n            key=lambda item: (-item.score, item.entity.rel_path, item.entity.start_line)\n        )\n        ranked_files = sorted(file_scores.items(), key=lambda kv: (-kv[1], kv[0]))\n\n        seen_test_files: set[str] = set()\n        ordered_tests: list[str] = []\n        for _, tpath in sorted(linked_tests, key=lambda x: (-x[0], x[1])):\n            if tpath not in seen_test_files:\n                seen_test_files.add(tpath)\n                ordered_tests.append(tpath)\n\n        return LocalizationResult(\n            ranked_entities=ranked[:top_k],\n            ranked_files=ranked_files[:top_k],\n            linked_tests=ordered_tests,\n            traceback_hits=tb_hits,\n            call_graph_edges={k: list(v) for k, v in self.call_graph.items() if v},\n            total_entities_indexed=len(self.entities),\n            total_files_indexed=self.file_count,\n        )\n', 'context_slicer.py': '"""Token-budgeted Minimal Context Slicing for Gemma 4 32K/128K context windows."""\n\nfrom __future__ import annotations\n\nimport ast\nfrom dataclasses import dataclass, field\nfrom pathlib import Path\n\nfrom .ast_localizer import ASTFaultLocalizer, CodeEntity, LocalizationResult\n\n\n@dataclass\nclass ContextSlice:\n    """Compact line-numbered repository slice tailored for Gemma 4 patch synthesis."""\n\n    prompt_context: str\n    estimated_tokens: int\n    included_entities: list[str] = field(default_factory=list)\n    included_files: list[str] = field(default_factory=list)\n    truncated: bool = False\n\n\ndef estimate_tokens(text: str) -> int:\n    """Conservative token estimator (~3.6 chars/token for Python code)."""\n    return max(1, (len(text) + 3) // 4)\n\n\ndef _format_numbered_lines(lines: list[str], start_line: int, end_line: int) -> str:\n    lo = max(1, start_line)\n    hi = min(len(lines), end_line)\n    out = []\n    for ln in range(lo, hi + 1):\n        out.append(f"L{ln:04d}: {lines[ln - 1]}")\n    return "\\n".join(out)\n\n\ndef _extract_import_block(lines: list[str], max_lines: int = 24) -> str:\n    """Extract top-level import header lines from a Python module."""\n    source = "\\n".join(lines)\n    try:\n        tree = ast.parse(source)\n    except SyntaxError:\n        return "\\n".join(\n            f"L{idx:04d}: {ln}"\n            for idx, ln in enumerate(lines[:max_lines], start=1)\n            if ln.strip().startswith(("import ", "from "))\n        )\n    import_lines: list[int] = []\n    for node in tree.body:\n        if isinstance(node, (ast.Import, ast.ImportFrom)):\n            end_ln = getattr(node, "end_lineno", node.lineno) or node.lineno\n            import_lines.extend(range(node.lineno, end_ln + 1))\n    if not import_lines:\n        return ""\n    picked = sorted(set(import_lines))[:max_lines]\n    return "\\n".join(\n        f"L{ln:04d}: {lines[ln - 1]}" for ln in picked if 1 <= ln <= len(lines)\n    )\n\n\nclass MinimalContextSlicer:\n    """Assemble minimal high-signal AST slices within strict token budgets."""\n\n    def __init__(self, repo_root: str | Path, localizer: ASTFaultLocalizer) -> None:\n        self.repo_root = Path(repo_root).resolve()\n        self.localizer = localizer\n        self._file_cache: dict[str, list[str]] = {}\n\n    def _get_lines(self, rel_path: str) -> list[str]:\n        if rel_path not in self._file_cache:\n            fp = self.repo_root / rel_path\n            if fp.is_file():\n                self._file_cache[rel_path] = fp.read_text(\n                    encoding="utf-8", errors="replace"\n                ).splitlines()\n            else:\n                self._file_cache[rel_path] = []\n        return self._file_cache[rel_path]\n\n    def build_slice(\n        self,\n        localization: LocalizationResult,\n        max_tokens: int = 2800,\n        max_focus_entities: int = 4,\n        include_neighbor_signatures: bool = True,\n        include_test_snippet: bool = True,\n    ) -> ContextSlice:\n        """Build a surgical context bundle for Gemma 4 without dumping whole files."""\n        sections: list[str] = []\n        included_entities: list[str] = []\n        included_files: list[str] = []\n        used_tokens = 0\n        truncated = False\n        seen_imports: set[str] = set()\n        seen_class_headers: set[tuple[str, str]] = set()\n\n        # Focus on concrete methods/functions first; fallback to classes if needed\n        candidates = [\n            item for item in localization.ranked_entities if item.entity.kind != "class"\n        ]\n        if not candidates:\n            candidates = list(localization.ranked_entities)\n\n        for susp in candidates[:max_focus_entities]:\n            ent: CodeEntity = susp.entity\n            lines = self._get_lines(ent.rel_path)\n            if not lines:\n                continue\n\n            parts: list[str] = [\n                f"### TARGET `{ent.rel_path}` :: `{ent.qualname}` "\n                f"(score={susp.score:.2f}, lines {ent.start_line}-{ent.end_line})"\n            ]\n            if susp.reasons:\n                parts.append(f"# localization_evidence: {\'; \'.join(susp.reasons)}")\n\n            if ent.rel_path not in seen_imports:\n                imp = _extract_import_block(lines)\n                if imp:\n                    parts.append("# Module imports:\\n" + imp)\n                seen_imports.add(ent.rel_path)\n\n            # Include full body of focal AST node (+1 line margin if small)\n            pad_start = max(1, ent.start_line - 1)\n            pad_end = min(len(lines), ent.end_line + 1)\n\n            if (\n                ent.parent_class\n                and (ent.rel_path, ent.parent_class) not in seen_class_headers\n            ):\n                cls_entities = [\n                    e\n                    for e in self.localizer.file_entities.get(ent.rel_path, ())\n                    if e.kind == "class" and e.name == ent.parent_class\n                ]\n                if cls_entities:\n                    cls_ent = cls_entities[0]\n                    if cls_ent.start_line < pad_start:\n                        parts.append(\n                            f"L{cls_ent.start_line:04d}: {lines[cls_ent.start_line - 1]}"\n                        )\n                seen_class_headers.add((ent.rel_path, ent.parent_class))\n\n            parts.append(_format_numbered_lines(lines, pad_start, pad_end))\n\n            if include_neighbor_signatures:\n                neighbor_lines: list[str] = []\n                for callee_id in self.localizer.call_graph.get(ent.entity_id, ())[:4]:\n                    callee = self.localizer.entities.get(callee_id)\n                    if callee:\n                        doc = f" # {callee.docstring}" if callee.docstring else ""\n                        neighbor_lines.append(\n                            f"  -> calls {callee.rel_path}:{callee.start_line} `{callee.signature}`{doc}"\n                        )\n                for caller_id in self.localizer.reverse_graph.get(ent.entity_id, ())[\n                    :3\n                ]:\n                    caller = self.localizer.entities.get(caller_id)\n                    if caller and not caller.is_test:\n                        neighbor_lines.append(\n                            f"  <- called_by {caller.rel_path}:{caller.start_line} `{caller.signature}`"\n                        )\n                if neighbor_lines:\n                    parts.append("# 1-hop AST neighbors:\\n" + "\\n".join(neighbor_lines))\n\n            block = "\\n".join(parts)\n            block_tokens = estimate_tokens(block)\n            if used_tokens + block_tokens > max_tokens and sections:\n                truncated = True\n                break\n            sections.append(block)\n            used_tokens += block_tokens\n            included_entities.append(ent.entity_id)\n            if ent.rel_path not in included_files:\n                included_files.append(ent.rel_path)\n\n        # Add compact failing test / traceback slice if budget remains\n        if (\n            include_test_snippet\n            and localization.linked_tests\n            and used_tokens < int(max_tokens * 0.85)\n        ):\n            test_rel = localization.linked_tests[0]\n            tlines = self._get_lines(test_rel)\n            if tlines:\n                snippet_lines = tlines[:45]\n                tblock = (\n                    f"### LINKED TEST (READ-ONLY — DO NOT EDIT) `{test_rel}`\\n"\n                    + _format_numbered_lines(snippet_lines, 1, len(snippet_lines))\n                )\n                ttok = estimate_tokens(tblock)\n                if used_tokens + ttok <= max_tokens:\n                    sections.append(tblock)\n                    used_tokens += ttok\n                    if test_rel not in included_files:\n                        included_files.append(test_rel)\n                else:\n                    truncated = True\n\n        prompt_context = "\\n\\n".join(sections)\n        return ContextSlice(\n            prompt_context=prompt_context,\n            estimated_tokens=estimate_tokens(prompt_context) if prompt_context else 0,\n            included_entities=included_entities,\n            included_files=included_files,\n            truncated=truncated,\n        )\n', 'diff_engine.py': '"""Unified diff & SEARCH/REPLACE parser, atomic hunk applier, AST syntax guard, and blast-radius metrics."""\n\nfrom __future__ import annotations\n\nimport ast\nimport difflib\nimport hashlib\nimport re\nfrom dataclasses import dataclass, field\nfrom pathlib import Path\nfrom typing import Iterable\n\nfrom .ast_localizer import is_test_path\n\nHUNK_HEADER_RE = re.compile(r"^@@\\s+-(\\d+)(?:,(\\d+))?\\s+\\+(\\d+)(?:,(\\d+))?\\s+@@(.*)$")\nSEARCH_REPLACE_RE = re.compile(\n    r"(?:^|\\n)(?:FILE:\\s*(?P<file>[^\\n]+)\\n)?<<<\\s*SEARCH\\s*\\n(?P<search>.*?)\\n===\\s*\\n(?P<replace>.*?)\\n>>>\\s*REPLACE",\n    re.DOTALL,\n)\n\n\n@dataclass\nclass Hunk:\n    """Single unified diff hunk."""\n\n    old_start: int\n    old_count: int\n    new_start: int\n    new_count: int\n    lines: list[str]  # prefixed with \' \', \'-\', or \'+\'\n    header_context: str = ""\n\n    @property\n    def expected_old_lines(self) -> list[str]:\n        return [ln[1:] for ln in self.lines if ln.startswith((" ", "-"))]\n\n    @property\n    def replacement_lines(self) -> list[str]:\n        return [ln[1:] for ln in self.lines if ln.startswith((" ", "+"))]\n\n    @property\n    def added_count(self) -> int:\n        return sum(1 for ln in self.lines if ln.startswith("+"))\n\n    @property\n    def removed_count(self) -> int:\n        return sum(1 for ln in self.lines if ln.startswith("-"))\n\n\n@dataclass\nclass FileDiff:\n    """Parsed unified diff for a single file."""\n\n    old_path: str\n    new_path: str\n    hunks: list[Hunk] = field(default_factory=list)\n    is_new_file: bool = False\n    is_deleted: bool = False\n\n    @property\n    def target_path(self) -> str:\n        if self.new_path and self.new_path != "/dev/null":\n            return self.new_path\n        return self.old_path\n\n\n@dataclass(frozen=True)\nclass BlastRadius:\n    """Quantitative size & risk footprint of a candidate patch."""\n\n    files_touched: int\n    hunks_count: int\n    lines_added: int\n    lines_removed: int\n    net_churn: int\n    touches_tests: bool\n    modified_files: tuple[str, ...]\n\n    @property\n    def penalty(self) -> float:\n        return round(\n            (self.files_touched * 1.5)\n            + (self.hunks_count * 0.6)\n            + ((self.lines_added + self.lines_removed) * 0.08),\n            6,\n        )\n\n\n@dataclass\nclass DiffApplyResult:\n    """Result of validating and applying a unified diff to an in-memory or on-disk repo."""\n\n    success: bool\n    updated_files: dict[str, str] = field(default_factory=dict)\n    errors: list[str] = field(default_factory=list)\n    syntax_valid: bool = False\n    blast_radius: BlastRadius | None = None\n    canonical_diff: str = ""\n\n\ndef _clean_diff_path(raw: str) -> str:\n    raw = raw.strip()\n    if raw == "/dev/null":\n        return "/dev/null"\n    if raw.startswith(("a/", "b/")):\n        raw = raw[2:]\n    return raw.replace("\\\\", "/").lstrip("./")\n\n\ndef _validate_safe_rel_path(rel_path: str) -> str:\n    if not rel_path or rel_path == "/dev/null":\n        raise ValueError(f"Invalid relative diff path: {rel_path!r}")\n    p = Path(rel_path)\n    if p.is_absolute() or ".." in p.parts:\n        raise ValueError(f"Path traversal blocked in diff: {rel_path!r}")\n    return p.as_posix()\n\n\ndef parse_unified_diff(diff_text: str) -> list[FileDiff]:\n    """Parse standard unified/git diff text into FileDiff objects."""\n    lines = diff_text.splitlines()\n    file_diffs: list[FileDiff] = []\n    current_file: FileDiff | None = None\n    current_hunk: Hunk | None = None\n    pending_old: str | None = None\n\n    i = 0\n    while i < len(lines):\n        line = lines[i]\n        if line.startswith("diff --git "):\n            pending_old = None\n            current_hunk = None\n            i += 1\n            continue\n        if line.startswith("--- "):\n            pending_old = _clean_diff_path(line[4:].split("\\t")[0])\n            current_hunk = None\n            i += 1\n            continue\n        if line.startswith("+++ "):\n            new_path = _clean_diff_path(line[4:].split("\\t")[0])\n            old_path = pending_old if pending_old is not None else new_path\n            current_file = FileDiff(\n                old_path=old_path,\n                new_path=new_path,\n                is_new_file=(old_path == "/dev/null"),\n                is_deleted=(new_path == "/dev/null"),\n            )\n            file_diffs.append(current_file)\n            current_hunk = None\n            pending_old = None\n            i += 1\n            continue\n\n        m = HUNK_HEADER_RE.match(line)\n        if m:\n            if current_file is None:\n                raise ValueError(\n                    f"Hunk header found before file header at line {i + 1}: {line}"\n                )\n            old_start = int(m.group(1))\n            old_count = int(m.group(2)) if m.group(2) is not None else 1\n            new_start = int(m.group(3))\n            new_count = int(m.group(4)) if m.group(4) is not None else 1\n            current_hunk = Hunk(\n                old_start=old_start,\n                old_count=old_count,\n                new_start=new_start,\n                new_count=new_count,\n                lines=[],\n                header_context=m.group(5).strip(),\n            )\n            current_file.hunks.append(current_hunk)\n            i += 1\n            continue\n\n        if current_hunk is not None:\n            if line.startswith((" ", "-", "+")):\n                current_hunk.lines.append(line)\n            elif line.startswith("\\\\ No newline at end of file"):\n                pass\n            elif line == "" and (\n                len(current_hunk.expected_old_lines) < current_hunk.old_count\n                or len(current_hunk.replacement_lines) < current_hunk.new_count\n            ):\n                # Tolerate empty line whose leading space was stripped by markdown formatting\n                current_hunk.lines.append(" ")\n            else:\n                current_hunk = None\n        i += 1\n\n    return [fd for fd in file_diffs if fd.hunks]\n\n\ndef parse_search_replace_blocks(\n    text: str, default_file: str | None = None\n) -> list[tuple[str, str, str]]:\n    """Extract (file_path, search_block, replace_block) tuples from Gemma 4 edit blocks."""\n    results: list[tuple[str, str, str]] = []\n    for match in SEARCH_REPLACE_RE.finditer(text):\n        fpath = (match.group("file") or default_file or "").strip()\n        if not fpath:\n            raise ValueError("SEARCH/REPLACE block missing FILE path")\n        results.append(\n            (_clean_diff_path(fpath), match.group("search"), match.group("replace"))\n        )\n    return results\n\n\ndef search_replace_to_unified_diff(\n    repo_root: str | Path,\n    edits: list[tuple[str, str, str]],\n) -> str:\n    """Convert exact SEARCH/REPLACE edits against files in repo_root into canonical unified diff."""\n    root = Path(repo_root).resolve()\n    grouped: dict[str, list[tuple[str, str]]] = {}\n    for rel, search, replace in edits:\n        safe_rel = _validate_safe_rel_path(rel)\n        grouped.setdefault(safe_rel, []).append((search, replace))\n\n    diff_chunks: list[str] = []\n    for rel, file_edits in sorted(grouped.items()):\n        fpath = root / rel\n        if not fpath.is_file():\n            raise FileNotFoundError(f"Target file not found for edit: {rel}")\n        orig_text = fpath.read_text(encoding="utf-8", errors="replace")\n        new_text = orig_text\n        for search, replace in file_edits:\n            count = new_text.count(search)\n            if count == 0:\n                raise ValueError(f"SEARCH block not found in {rel}")\n            if count > 1:\n                raise ValueError(f"Ambiguous SEARCH block ({count} matches) in {rel}")\n            new_text = new_text.replace(search, replace, 1)\n\n        orig_lines = orig_text.splitlines(keepends=True)\n        new_lines = new_text.splitlines(keepends=True)\n        udiff = list(\n            difflib.unified_diff(\n                orig_lines,\n                new_lines,\n                fromfile=f"a/{rel}",\n                tofile=f"b/{rel}",\n                n=3,\n            )\n        )\n        if udiff:\n            diff_chunks.append("".join(udiff))\n    return "\\n".join(diff_chunks)\n\n\ndef compute_blast_radius(file_diffs: Iterable[FileDiff]) -> BlastRadius:\n    """Compute files touched, hunk counts, line churn, and test-touch guard flag."""\n    diffs = list(file_diffs)\n    files: list[str] = []\n    hunks_count = 0\n    added = 0\n    removed = 0\n    touches_tests = False\n\n    for fd in diffs:\n        tpath = fd.target_path\n        files.append(tpath)\n        if is_test_path(tpath):\n            touches_tests = True\n        for h in fd.hunks:\n            hunks_count += 1\n            added += h.added_count\n            removed += h.removed_count\n\n    return BlastRadius(\n        files_touched=len(set(files)),\n        hunks_count=hunks_count,\n        lines_added=added,\n        lines_removed=removed,\n        net_churn=abs(added - removed),\n        touches_tests=touches_tests,\n        modified_files=tuple(sorted(set(files))),\n    )\n\n\ndef validate_python_syntax(rel_path: str, content: str) -> str | None:\n    """Return error string if Python file has SyntaxError, else None."""\n    if not rel_path.endswith(".py"):\n        return None\n    try:\n        ast.parse(content, filename=rel_path)\n        return None\n    except SyntaxError as exc:\n        return f"{rel_path}:{exc.lineno}: {exc.msg}"\n\n\ndef _locate_hunk_index(\n    file_lines: list[str], hunk: Hunk, max_drift: int = 20\n) -> int | None:\n    """Find deterministic 0-based index where hunk.expected_old_lines matches verbatim."""\n    expected = hunk.expected_old_lines\n    if not expected:\n        return max(0, min(len(file_lines), hunk.old_start - 1))\n\n    exp_len = len(expected)\n    nominal_idx = max(0, hunk.old_start - 1)\n    if (\n        nominal_idx + exp_len <= len(file_lines)\n        and file_lines[nominal_idx : nominal_idx + exp_len] == expected\n    ):\n        return nominal_idx\n\n    # Search bounded window around nominal_idx; accept only if unique match\n    lo = max(0, nominal_idx - max_drift)\n    hi = min(len(file_lines) - exp_len, nominal_idx + max_drift)\n    matches = [\n        idx for idx in range(lo, hi + 1) if file_lines[idx : idx + exp_len] == expected\n    ]\n    if len(matches) == 1:\n        return matches[0]\n    return None\n\n\ndef apply_unified_diff(\n    repo_root: str | Path,\n    diff_text: str,\n    write_to_disk: bool = False,\n    allow_test_edits: bool = False,\n) -> DiffApplyResult:\n    """Validate, apply, and AST-check a unified diff without leaving partial edits on error."""\n    root = Path(repo_root).resolve()\n    try:\n        file_diffs = parse_unified_diff(diff_text)\n    except ValueError as exc:\n        return DiffApplyResult(success=False, errors=[f"Diff parse error: {exc}"])\n\n    if not file_diffs:\n        return DiffApplyResult(success=False, errors=["Empty or unparseable diff"])\n\n    blast = compute_blast_radius(file_diffs)\n    if not allow_test_edits and blast.touches_tests:\n        return DiffApplyResult(\n            success=False,\n            errors=["Policy violation: candidate patch modifies test files"],\n            blast_radius=blast,\n        )\n\n    updated_files: dict[str, str] = {}\n    canonical_parts: list[str] = []\n    errors: list[str] = []\n\n    for fd in file_diffs:\n        try:\n            rel_path = _validate_safe_rel_path(fd.target_path)\n        except ValueError as exc:\n            errors.append(str(exc))\n            continue\n\n        fpath = root / rel_path\n        if fd.is_new_file:\n            orig_lines: list[str] = []\n            had_trailing_newline = True\n        else:\n            if not fpath.is_file():\n                errors.append(f"Target file does not exist: {rel_path}")\n                continue\n            raw_text = fpath.read_text(encoding="utf-8", errors="replace")\n            had_trailing_newline = raw_text.endswith("\\n") or raw_text == ""\n            orig_lines = raw_text.splitlines()\n\n        new_lines = list(orig_lines)\n        offset_shift = 0\n        hunk_failed = False\n\n        for h_idx, hunk in enumerate(fd.hunks):\n            if fd.is_new_file:\n                new_lines.extend(hunk.replacement_lines)\n                continue\n            shifted_hunk = Hunk(\n                old_start=hunk.old_start + offset_shift,\n                old_count=hunk.old_count,\n                new_start=hunk.new_start,\n                new_count=hunk.new_count,\n                lines=hunk.lines,\n            )\n            idx = _locate_hunk_index(new_lines, shifted_hunk)\n            if idx is None:\n                errors.append(\n                    f"Hunk #{h_idx + 1} mismatch in {rel_path} near line {hunk.old_start}"\n                )\n                hunk_failed = True\n                break\n            exp_len = len(hunk.expected_old_lines)\n            rep = hunk.replacement_lines\n            new_lines[idx : idx + exp_len] = rep\n            offset_shift += len(rep) - exp_len\n\n        if hunk_failed:\n            continue\n\n        new_content = "\\n".join(new_lines)\n        if had_trailing_newline and new_lines:\n            new_content += "\\n"\n\n        syntax_err = validate_python_syntax(rel_path, new_content)\n        if syntax_err:\n            errors.append(f"AST syntax check failed: {syntax_err}")\n            continue\n\n        updated_files[rel_path] = new_content\n        old_for_canon = (\n            "\\n".join(orig_lines) + ("\\n" if orig_lines else "")\n        ).splitlines(keepends=True)\n        new_for_canon = new_content.splitlines(keepends=True)\n        canonical_parts.append(\n            "".join(\n                difflib.unified_diff(\n                    old_for_canon,\n                    new_for_canon,\n                    fromfile=f"a/{rel_path}",\n                    tofile=f"b/{rel_path}",\n                    n=3,\n                )\n            )\n        )\n\n    if errors:\n        return DiffApplyResult(\n            success=False,\n            errors=errors,\n            syntax_valid=False,\n            blast_radius=blast,\n        )\n\n    canonical_diff = "\\n".join(p for p in canonical_parts if p)\n    if write_to_disk:\n        for rel_path, content in updated_files.items():\n            target = root / rel_path\n            target.parent.mkdir(parents=True, exist_ok=True)\n            target.write_text(content, encoding="utf-8")\n\n    return DiffApplyResult(\n        success=True,\n        updated_files=updated_files,\n        errors=[],\n        syntax_valid=True,\n        blast_radius=blast,\n        canonical_diff=canonical_diff,\n    )\n\n\ndef diff_sha256(canonical_diff: str) -> str:\n    """Stable content hash for deterministic tie-breaking."""\n    return hashlib.sha256(canonical_diff.encode("utf-8")).hexdigest()\n', 'patch_pipeline.py': '"""Multi-candidate patch synthesis, test-execution self-refinement, regression guard, and deterministic ranking."""\n\nfrom __future__ import annotations\n\nimport os\nimport re\nimport shutil\nimport subprocess\nimport sys\nimport tempfile\nimport time\nfrom dataclasses import dataclass, field\nfrom pathlib import Path\nfrom typing import Sequence\n\nfrom .ast_localizer import ASTFaultLocalizer, LocalizationResult\nfrom .diff_engine import BlastRadius, DiffApplyResult, apply_unified_diff, diff_sha256\n\nUNITTEST_RAN_RE = re.compile(r"Ran\\s+(\\d+)\\s+tests?\\s+in")\nPYTEST_PASSED_RE = re.compile(r"(\\d+)\\s+passed")\nPYTEST_FAILED_RE = re.compile(r"(\\d+)\\s+failed")\nERROR_EXC_RE = re.compile(r"([A-Za-z_][A-Za-z0-9_]*(?:Error|Exception))\\s*:\\s*(.+)")\n\n\n@dataclass\nclass CandidatePatch:\n    """Single patch candidate synthesized by a Gemma 4 strategy/temperature arm."""\n\n    candidate_id: str\n    strategy: str\n    diff_text: str\n    temperature: float = 0.0\n    refine_round: int = 0\n    raw_thoughts: str = ""\n\n\n@dataclass\nclass TestExecutionReport:\n    """Structured test execution & traceback summary from sandbox validation."""\n\n    passed: bool\n    returncode: int\n    total_run: int\n    failed_count: int\n    stdout: str\n    stderr: str\n    duration_sec: float\n    failure_summary: str = ""\n    timed_out: bool = False\n\n\n@dataclass\nclass CandidateEvaluation:\n    """Complete evaluation & ranking record for one candidate patch."""\n\n    candidate: CandidatePatch\n    apply_result: DiffApplyResult\n    target_report: TestExecutionReport | None\n    regression_report: TestExecutionReport | None\n    policy_compliant: bool\n    localization_aligned: bool\n    rank_score: float\n    rank_tuple: tuple\n    canonical_hash: str\n    rejection_reason: str = ""\n\n\ndef _parse_failure_summary(stdout: str, stderr: str) -> str:\n    combined = f"{stdout}\\n{stderr}".strip()\n    lines = [ln for ln in combined.splitlines() if ln.strip()]\n    exc_hits = ERROR_EXC_RE.findall(combined)\n    exc_line = f"{exc_hits[-1][0]}: {exc_hits[-1][1]}" if exc_hits else ""\n    tb_tail = "\\n".join(lines[-10:])\n    if exc_line and exc_line not in tb_tail:\n        return f"{exc_line}\\n{tb_tail}"\n    return tb_tail[:900]\n\n\ndef _parse_test_counts(stdout: str, stderr: str, passed: bool) -> tuple[int, int]:\n    combined = f"{stdout}\\n{stderr}"\n    m_unit = UNITTEST_RAN_RE.search(combined)\n    if m_unit:\n        total = int(m_unit.group(1))\n        failed = (\n            0 if passed else max(1, combined.count("FAIL:") + combined.count("ERROR:"))\n        )\n        return total, min(total, failed)\n    m_p = PYTEST_PASSED_RE.search(combined)\n    m_f = PYTEST_FAILED_RE.search(combined)\n    p_cnt = int(m_p.group(1)) if m_p else 0\n    f_cnt = int(m_f.group(1)) if m_f else (0 if passed else 1)\n    total = p_cnt + f_cnt\n    return max(total, 1 if not passed else total), f_cnt\n\n\ndef _ignore_copy(_dir: str, names: list[str]) -> list[str]:\n    return [\n        n\n        for n in names\n        if n in {".git", "__pycache__", ".pytest_cache", ".venv", "venv"}\n    ]\n\n\nclass RegressionGuard:\n    """Runs target + regression tests inside an isolated sandbox copy with strict anti-tamper checks."""\n\n    def __init__(\n        self,\n        repo_root: str | Path,\n        allow_test_edits: bool = False,\n        timeout_seconds: int = 30,\n        python_executable: str | None = None,\n    ) -> None:\n        self.repo_root = Path(repo_root).resolve()\n        self.allow_test_edits = allow_test_edits\n        self.timeout_seconds = timeout_seconds\n        self.python_exe = python_executable or sys.executable\n\n    def _run_suite_in_dir(\n        self, workdir: Path, cmd: Sequence[str]\n    ) -> TestExecutionReport:\n        t0 = time.perf_counter()\n        env = os.environ.copy()\n        env["PYTHONDONTWRITEBYTECODE"] = "1"\n        try:\n            proc = subprocess.run(\n                list(cmd),\n                cwd=str(workdir),\n                env=env,\n                capture_output=True,\n                text=True,\n                encoding="utf-8",\n                errors="replace",\n                timeout=self.timeout_seconds,\n            )\n            dur = round(time.perf_counter() - t0, 4)\n            passed = proc.returncode == 0\n            total, failed = _parse_test_counts(proc.stdout, proc.stderr, passed)\n            summary = "" if passed else _parse_failure_summary(proc.stdout, proc.stderr)\n            return TestExecutionReport(\n                passed=passed,\n                returncode=proc.returncode,\n                total_run=total,\n                failed_count=failed,\n                stdout=proc.stdout,\n                stderr=proc.stderr,\n                duration_sec=dur,\n                failure_summary=summary,\n                timed_out=False,\n            )\n        except subprocess.TimeoutExpired as exc:\n            dur = round(time.perf_counter() - t0, 4)\n            return TestExecutionReport(\n                passed=False,\n                returncode=124,\n                total_run=1,\n                failed_count=1,\n                stdout=str(exc.stdout or ""),\n                stderr=str(exc.stderr or ""),\n                duration_sec=dur,\n                failure_summary=f"TimeoutExpired after {self.timeout_seconds}s",\n                timed_out=True,\n            )\n\n    def evaluate_candidate(\n        self,\n        candidate: CandidatePatch,\n        target_cmd: Sequence[str],\n        regression_cmd: Sequence[str] | None = None,\n        localization: LocalizationResult | None = None,\n    ) -> CandidateEvaluation:\n        """Apply candidate to isolated temp sandbox, validate syntax/policy, and run tests."""\n        apply_res = apply_unified_diff(\n            self.repo_root,\n            candidate.diff_text,\n            write_to_disk=False,\n            allow_test_edits=self.allow_test_edits,\n        )\n\n        if not apply_res.success or not apply_res.syntax_valid:\n            reason = (\n                "; ".join(apply_res.errors)\n                or "Diff application or AST syntax check failed"\n            )\n            policy_ok = not (\n                apply_res.blast_radius\n                and apply_res.blast_radius.touches_tests\n                and not self.allow_test_edits\n            )\n            return CandidateEvaluation(\n                candidate=candidate,\n                apply_result=apply_res,\n                target_report=None,\n                regression_report=None,\n                policy_compliant=policy_ok,\n                localization_aligned=False,\n                rank_score=-1000.0,\n                rank_tuple=(0, 0, 0, 0, 0.0, 0, -9999.0, ""),\n                canonical_hash=diff_sha256(candidate.diff_text),\n                rejection_reason=reason,\n            )\n\n        blast = apply_res.blast_radius\n        policy_ok = bool(\n            blast is not None and (self.allow_test_edits or not blast.touches_tests)\n        )\n\n        # Check alignment with top-3 localized production files\n        loc_aligned = False\n        if localization and localization.ranked_files and blast:\n            top_files = {f for f, _ in localization.ranked_files[:3]}\n            loc_aligned = bool(top_files & set(blast.modified_files))\n\n        with tempfile.TemporaryDirectory(prefix="gemma4_sandbox_") as tmp:\n            sandbox_dir = Path(tmp) / "repo"\n            shutil.copytree(self.repo_root, sandbox_dir, ignore=_ignore_copy)\n            for rel_path, new_content in apply_res.updated_files.items():\n                dst = sandbox_dir / rel_path\n                dst.parent.mkdir(parents=True, exist_ok=True)\n                dst.write_text(new_content, encoding="utf-8")\n\n            target_rep = self._run_suite_in_dir(sandbox_dir, target_cmd)\n            if regression_cmd is not None:\n                reg_rep = self._run_suite_in_dir(sandbox_dir, regression_cmd)\n            else:\n                reg_rep = target_rep\n\n        return DeterministicPatchRanker.score_evaluation(\n            candidate=candidate,\n            apply_result=apply_res,\n            target_report=target_rep,\n            regression_report=reg_rep,\n            policy_compliant=policy_ok,\n            localization_aligned=loc_aligned,\n        )\n\n\nclass DeterministicPatchRanker:\n    """Deterministic lexicographic + composite patch selector with SHA-256 tie-break."""\n\n    @staticmethod\n    def score_evaluation(\n        candidate: CandidatePatch,\n        apply_result: DiffApplyResult,\n        target_report: TestExecutionReport | None,\n        regression_report: TestExecutionReport | None,\n        policy_compliant: bool,\n        localization_aligned: bool,\n    ) -> CandidateEvaluation:\n        blast: BlastRadius | None = apply_result.blast_radius\n        blast_penalty = blast.penalty if blast else 999.0\n        canon_hash = diff_sha256(apply_result.canonical_diff or candidate.diff_text)\n\n        t_pass = bool(target_report and target_report.passed)\n        r_pass = bool(regression_report and regression_report.passed)\n\n        total_tests = 0\n        passed_tests = 0\n        if target_report:\n            total_tests += target_report.total_run\n            passed_tests += max(0, target_report.total_run - target_report.failed_count)\n        if regression_report and regression_report is not target_report:\n            total_tests += regression_report.total_run\n            passed_tests += max(\n                0, regression_report.total_run - regression_report.failed_count\n            )\n        pass_ratio = round(passed_tests / max(1, total_tests), 6)\n\n        # Lexicographic priority guarantees zero-regression + target-pass beats smaller partial fixes\n        # Negate hex prefix int so smaller hash wins deterministically if literally every metric ties\n        hash_tie = -int(canon_hash[:12], 16)\n        rank_tuple = (\n            1 if policy_compliant else 0,\n            1 if apply_result.syntax_valid else 0,\n            1 if t_pass else 0,\n            1 if r_pass else 0,\n            pass_ratio,\n            1 if localization_aligned else 0,\n            round(-blast_penalty, 6),\n            hash_tie,\n        )\n\n        composite = (\n            (1000.0 if policy_compliant else -1000.0)\n            + (200.0 if apply_result.syntax_valid else -500.0)\n            + (500.0 if t_pass else 0.0)\n            + (400.0 if r_pass else -250.0)\n            + (100.0 * pass_ratio)\n            + (25.0 if localization_aligned else 0.0)\n            - blast_penalty\n        )\n\n        rejection = ""\n        if not policy_compliant:\n            rejection = "Modified forbidden test file"\n        elif not apply_result.syntax_valid:\n            rejection = "; ".join(apply_result.errors)\n        elif not t_pass:\n            rejection = f"Target tests failed: {target_report.failure_summary if target_report else \'no run\'}"\n        elif not r_pass:\n            rejection = f"Regression tests failed: {regression_report.failure_summary if regression_report else \'no run\'}"\n\n        return CandidateEvaluation(\n            candidate=candidate,\n            apply_result=apply_result,\n            target_report=target_report,\n            regression_report=regression_report,\n            policy_compliant=policy_compliant,\n            localization_aligned=localization_aligned,\n            rank_score=round(composite, 4),\n            rank_tuple=rank_tuple,\n            canonical_hash=canon_hash,\n            rejection_reason=rejection,\n        )\n\n    @staticmethod\n    def rank(evaluations: Sequence[CandidateEvaluation]) -> list[CandidateEvaluation]:\n        """Sort candidates strictly from best to worst."""\n        return sorted(evaluations, key=lambda ev: ev.rank_tuple, reverse=True)\n', 'gemma4_runtime.py': '"""Gemma 4 runtime adapters: QAT vLLM, Transformers+PEFT, channel-thought stripping, and CPU deterministic benchmark runtime."""\n\nfrom __future__ import annotations\n\nimport json\nimport re\nimport urllib.request\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Any, Protocol\n\nfrom .context_slicer import ContextSlice\nfrom .diff_engine import parse_search_replace_blocks, search_replace_to_unified_diff\n\nTHOUGHT_CHANNEL_RE = re.compile(r"<\\|channel>thought\\b.*?<channel\\|>", re.DOTALL)\nTHOUGHT_TAG_RE = re.compile(r"<thought>.*?</thought>", re.DOTALL)\nFENCED_DIFF_RE = re.compile(r"```(?:diff|patch)?\\s*\\n(.*?)```", re.DOTALL)\n\n\ndef strip_gemma4_thoughts(raw_output: str) -> tuple[str, str]:\n    """Separate Gemma 4 internal thought channel from user/tool output cleanly."""\n    thoughts: list[str] = []\n    for regex in (THOUGHT_CHANNEL_RE, THOUGHT_TAG_RE):\n        for m in regex.findall(raw_output):\n            thoughts.append(m.strip())\n    cleaned = THOUGHT_CHANNEL_RE.sub("", raw_output)\n    cleaned = THOUGHT_TAG_RE.sub("", cleaned).strip()\n    return cleaned, "\\n\\n".join(thoughts)\n\n\ndef extract_diff_payload(text: str, repo_root: str | None = None) -> str:\n    """Extract unified diff or SEARCH/REPLACE edit from Gemma 4 model output."""\n    cleaned, _ = strip_gemma4_thoughts(text)\n    if "<<<SEARCH" in cleaned and repo_root is not None:\n        edits = parse_search_replace_blocks(cleaned)\n        if edits:\n            return search_replace_to_unified_diff(repo_root, edits)\n\n    for block in FENCED_DIFF_RE.findall(cleaned):\n        if "--- " in block and "+++ " in block and "@@ " in block:\n            return block.strip() + "\\n"\n\n    if "--- " in cleaned and "+++ " in cleaned and "@@ " in cleaned:\n        idx = cleaned.find("--- ")\n        return cleaned[idx:].strip() + "\\n"\n    return ""\n\n\n@dataclass\nclass SynthesisProposal:\n    """Single patch generation output from a runtime backend."""\n\n    strategy: str\n    diff_text: str\n    temperature: float\n    raw_thoughts: str = ""\n\n\nclass RuntimeBackend(Protocol):\n    """Protocol implemented by vLLM, HF Transformers, and DeterministicRuleRuntime."""\n\n    def synthesize_candidates(\n        self,\n        repo_root: str,\n        issue_text: str,\n        context_slice: ContextSlice,\n        failure_feedback: str = "",\n        round_idx: int = 0,\n    ) -> list[SynthesisProposal]: ...\n\n\nclass Gemma4VLLMRuntime:\n    """Client for Gemma 4 QAT (`w4a16-ct`) served via vLLM OpenAI endpoint on Kaggle L4/H100."""\n\n    def __init__(\n        self,\n        base_url: str = "http://127.0.0.1:8000/v1",\n        model_id: str = "gemma-4-31b-it-qat-w4a16-ct",\n        max_tokens: int = 2048,\n        timeout_seconds: int = 120,\n    ) -> None:\n        self.base_url = base_url.rstrip("/")\n        self.model_id = model_id\n        self.max_tokens = max_tokens\n        self.timeout_seconds = timeout_seconds\n\n    def _chat(self, messages: list[dict[str, str]], temperature: float) -> str:\n        payload = {\n            "model": self.model_id,\n            "messages": messages,\n            "temperature": temperature,\n            "top_p": 0.95,\n            "top_k": 64,\n            "max_tokens": self.max_tokens,\n        }\n        req = urllib.request.Request(\n            f"{self.base_url}/chat/completions",\n            data=json.dumps(payload).encode("utf-8"),\n            headers={"Content-Type": "application/json"},\n            method="POST",\n        )\n        with urllib.request.urlopen(req, timeout=self.timeout_seconds) as resp:\n            data = json.loads(resp.read().decode("utf-8"))\n        return data["choices"][0]["message"]["content"] or ""\n\n    def synthesize_candidates(\n        self,\n        repo_root: str,\n        issue_text: str,\n        context_slice: ContextSlice,\n        failure_feedback: str = "",\n        round_idx: int = 0,\n    ) -> list[SynthesisProposal]:\n        arms = [\n            (\n                "surgical_greedy",\n                0.0,\n                "Produce the smallest possible unified diff fixing the root cause.",\n            ),\n            (\n                "boundary_guard",\n                0.35,\n                "Check boundary/off-by-one/None/empty-input invariants and emit a minimal unified diff.",\n            ),\n            (\n                "state_invariant",\n                0.65,\n                "Check state mutation, aliasing, and type invariants and emit a minimal unified diff.",\n            ),\n        ]\n        proposals: list[SynthesisProposal] = []\n        for strategy, temp, hint in arms:\n            sys_prompt = (\n                "You are an autonomous Gemma 4 SWE bug-fixing agent. "\n                "NEVER modify test files. Output ONLY a valid unified diff inside ```diff fences."\n            )\n            user_prompt = (\n                f"ISSUE:\\n{issue_text}\\n\\n"\n                f"AST CONTEXT SLICE:\\n{context_slice.prompt_context}\\n\\n"\n                f"STRATEGY HINT: {hint}\\n"\n            )\n            if failure_feedback:\n                user_prompt += f"\\nPREVIOUS TEST FAILURE DELTA (ROUND {round_idx}):\\n{failure_feedback}\\n"\n            raw = self._chat(\n                [\n                    {"role": "system", "content": sys_prompt},\n                    {"role": "user", "content": user_prompt},\n                ],\n                temperature=temp,\n            )\n            _, thoughts = strip_gemma4_thoughts(raw)\n            diff_txt = extract_diff_payload(raw, repo_root=repo_root)\n            if diff_txt:\n                proposals.append(\n                    SynthesisProposal(\n                        strategy=strategy,\n                        diff_text=diff_txt,\n                        temperature=temp,\n                        raw_thoughts=thoughts,\n                    )\n                )\n        return proposals\n\n\nclass Gemma4TransformersRuntime:\n    """In-process HuggingFace Transformers + PEFT LoRA inference wrapper for Gemma 4."""\n\n    def __init__(self, model: Any, tokenizer: Any, max_new_tokens: int = 1536) -> None:\n        self.model = model\n        self.tokenizer = tokenizer\n        self.max_new_tokens = max_new_tokens\n\n    def synthesize_candidates(\n        self,\n        repo_root: str,\n        issue_text: str,\n        context_slice: ContextSlice,\n        failure_feedback: str = "",\n        round_idx: int = 0,\n    ) -> list[SynthesisProposal]:\n        import torch\n\n        arms = [("hf_greedy", 0.0), ("hf_sampled", 0.35)]\n        proposals: list[SynthesisProposal] = []\n        prompt = (\n            f"Fix the following Python bug without modifying tests. Output ```diff unified diff.\\n\\n"\n            f"ISSUE:\\n{issue_text}\\n\\nCONTEXT:\\n{context_slice.prompt_context}\\n"\n        )\n        if failure_feedback:\n            prompt += f"\\nTEST FAILURE FEEDBACK:\\n{failure_feedback}\\n"\n        messages = [{"role": "user", "content": prompt}]\n        rendered = self.tokenizer.apply_chat_template(\n            messages, tokenize=False, add_generation_prompt=True\n        )\n        inputs = self.tokenizer(rendered, return_tensors="pt").to(self.model.device)\n        for strategy, temp in arms:\n            gen_kwargs: dict[str, Any] = {\n                "max_new_tokens": self.max_new_tokens,\n                "do_sample": temp > 0.0,\n            }\n            if temp > 0.0:\n                gen_kwargs["temperature"] = temp\n                gen_kwargs["top_p"] = 0.95\n                gen_kwargs["top_k"] = 64\n            with torch.inference_mode():\n                out_ids = self.model.generate(**inputs, **gen_kwargs)\n            gen_only = out_ids[0][inputs["input_ids"].shape[1] :]\n            raw = self.tokenizer.decode(gen_only, skip_special_tokens=False)\n            _, thoughts = strip_gemma4_thoughts(raw)\n            diff_txt = extract_diff_payload(raw, repo_root=repo_root)\n            if diff_txt:\n                proposals.append(\n                    SynthesisProposal(\n                        strategy=strategy,\n                        diff_text=diff_txt,\n                        temperature=temp,\n                        raw_thoughts=thoughts,\n                    )\n                )\n        return proposals\n\n\nclass DeterministicRuleRuntime:\n    """Zero-GPU candidate generator for local CPU verification of the agent loop & ranking pipeline.\n\n    Accepts injected candidate trajectories or derives targeted structural bug-fix hypotheses\n    (e.g., mutable default argument, off-by-one slice/comparison, zero-division/empty guard)\n    from the AST-localized context slice.\n    """\n\n    def __init__(\n        self, scripted_rounds: dict[int, list[SynthesisProposal]] | None = None\n    ) -> None:\n        self.scripted_rounds = scripted_rounds or {}\n\n    def synthesize_candidates(\n        self,\n        repo_root: str,\n        issue_text: str,\n        context_slice: ContextSlice,\n        failure_feedback: str = "",\n        round_idx: int = 0,\n    ) -> list[SynthesisProposal]:\n        if round_idx in self.scripted_rounds:\n            return list(self.scripted_rounds[round_idx])\n\n        proposals: list[SynthesisProposal] = []\n        # Synthesize structural hypotheses directly from the top localized files\n        for rel_path in context_slice.included_files:\n            if rel_path.startswith("tests/") or "test_" in rel_path:\n                continue\n            fpath = Path(repo_root) / rel_path\n            if not fpath.is_file():\n                continue\n            src = fpath.read_text(encoding="utf-8", errors="replace")\n\n            # Hypothesis A: Mutable default list/dict in def signature -> None default\n            m_mut = re.search(\n                r"(def\\s+\\w+\\([^)]*?)(\\w+)\\s*:\\s*[^=,\\)]+=\\s*(\\[\\]|\\{\\})|"\n                r"(def\\s+\\w+\\([^)]*?)(\\w+)\\s*=\\s*(\\[\\]|\\{\\})",\n                src,\n            )\n            if m_mut:\n                full_line_match = re.search(\n                    r"^([ \\t]*def\\s+\\w+\\([^\\n]*=\\s*(\\[\\]|\\{\\})[^\\n]*\\)[^\\n:]*:\\n)([ \\t]+)",\n                    src,\n                    re.MULTILINE,\n                )\n                if full_line_match:\n                    hdr = full_line_match.group(1)\n                    lit = full_line_match.group(2)\n                    indent = full_line_match.group(3)\n                    param_m = re.search(r"(\\w+)\\s*(?::[^,=()]+)?=\\s*(?:\\[\\]|\\{\\})", hdr)\n                    if param_m:\n                        pname = param_m.group(1)\n                        new_hdr = re.sub(\n                            rf"(\\b{re.escape(pname)}\\s*(?::[^,=()]+)?=)\\s*(?:\\[\\]|\\{{\\}})",\n                            r"\\1 None",\n                            hdr,\n                            count=1,\n                        )\n                        init_line = f"{indent}if {pname} is None:\\n{indent}    {pname} = {lit}\\n"\n                        udiff = search_replace_to_unified_diff(\n                            repo_root,\n                            [(rel_path, hdr + indent, new_hdr + init_line + indent)],\n                        )\n                        proposals.append(\n                            SynthesisProposal(\n                                strategy="mutable_default_guard",\n                                diff_text=udiff,\n                                temperature=0.0,\n                            )\n                        )\n\n            # Hypothesis B: Empty sequence guard before division by len(...)\n            if (\n                "ZeroDivisionError" in (issue_text + failure_feedback)\n                or "/ len(" in src\n            ):\n                m_div = re.search(\n                    r"^([ \\t]+)return\\s+sum\\((\\w+)\\)\\s*/\\s*len\\(\\2\\)\\s*$",\n                    src,\n                    re.MULTILINE,\n                )\n                if m_div:\n                    indent = m_div.group(1)\n                    var = m_div.group(2)\n                    old_blk = m_div.group(0)\n                    new_blk = (\n                        f"{indent}if not {var}:\\n{indent}    return 0.0\\n{old_blk}"\n                    )\n                    udiff = search_replace_to_unified_diff(\n                        repo_root, [(rel_path, old_blk, new_blk)]\n                    )\n                    proposals.append(\n                        SynthesisProposal(\n                            strategy="empty_division_guard",\n                            diff_text=udiff,\n                            temperature=0.2,\n                        )\n                    )\n\n            # Hypothesis C: Strict \'<\' vs \'<=\' inclusive upper bound / off-by-one candidate\n            m_cmp = re.search(\n                r"^([ \\t]+if\\s+[^:\\n]+?)<([^=][^:\\n]*:)$", src, re.MULTILINE\n            )\n            if m_cmp and (\n                "inclusive" in issue_text.lower()\n                or "boundary" in issue_text.lower()\n                or "AssertionError" in failure_feedback\n            ):\n                old_ln = m_cmp.group(0)\n                new_ln = f"{m_cmp.group(1)}<={m_cmp.group(2)}"\n                udiff = search_replace_to_unified_diff(\n                    repo_root, [(rel_path, old_ln, new_ln)]\n                )\n                proposals.append(\n                    SynthesisProposal(\n                        strategy="inclusive_boundary_fix",\n                        diff_text=udiff,\n                        temperature=0.35,\n                    )\n                )\n\n        return proposals\n', 'agent_orchestrator.py': '"""End-to-end 5-stage Autonomous SWE Agent orchestrator optimized for Gemma 4."""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, field\nfrom pathlib import Path\nfrom typing import Sequence\n\nfrom .ast_localizer import ASTFaultLocalizer, LocalizationResult\nfrom .context_slicer import ContextSlice, MinimalContextSlicer\nfrom .diff_engine import apply_unified_diff\nfrom .gemma4_runtime import DeterministicRuleRuntime, RuntimeBackend\nfrom .patch_pipeline import (\n    CandidateEvaluation,\n    CandidatePatch,\n    DeterministicPatchRanker,\n    RegressionGuard,\n)\n\n\n@dataclass\nclass AgentRunResult:\n    """Full structured trace and selected patch from AutonomousSWEAgent."""\n\n    solved: bool\n    selected_patch: str\n    selected_evaluation: CandidateEvaluation | None\n    ranked_candidates: list[CandidateEvaluation]\n    localization: LocalizationResult\n    context_slice: ContextSlice\n    rounds_executed: int\n    telemetry: dict[str, object] = field(default_factory=dict)\n\n\nclass AutonomousSWEAgent:\n    """5-Stage Gemma 4 SWE Bug-Fixing Agent:\n    1. AST Fault Localization (call graph + traceback + BM25)\n    2. Minimal Context Slicing (token-budgeted AST window)\n    3. Multi-Candidate Patch Synthesis (temperature/strategy arms)\n    4. Test-Execution Self-Refinement & Regression Guard (isolated sandbox + anti-test-tamper)\n    5. Deterministic Patch Ranking (lexicographic pass + zero-regression + blast radius + sha256)\n    """\n\n    def __init__(\n        self,\n        runtime: RuntimeBackend | None = None,\n        max_context_tokens: int = 2800,\n        max_refine_rounds: int = 2,\n        timeout_seconds: int = 30,\n        allow_test_edits: bool = False,\n    ) -> None:\n        self.runtime: RuntimeBackend = runtime or DeterministicRuleRuntime()\n        self.max_context_tokens = max_context_tokens\n        self.max_refine_rounds = max_refine_rounds\n        self.timeout_seconds = timeout_seconds\n        self.allow_test_edits = allow_test_edits\n\n    def solve_issue(\n        self,\n        repo_root: str | Path,\n        issue_text: str,\n        target_cmd: Sequence[str],\n        regression_cmd: Sequence[str] | None = None,\n        traceback_text: str = "",\n        apply_best: bool = False,\n    ) -> AgentRunResult:\n        root = Path(repo_root).resolve()\n\n        # Stage 1: AST Fault Localization\n        localizer = ASTFaultLocalizer(root)\n        localizer.index_repository()\n        localization = localizer.localize(\n            issue_text=issue_text,\n            traceback_text=traceback_text,\n            top_k=8,\n        )\n\n        # Stage 2: Minimal Context Slicing\n        slicer = MinimalContextSlicer(root, localizer)\n        ctx_slice = slicer.build_slice(\n            localization=localization,\n            max_tokens=self.max_context_tokens,\n            max_focus_entities=4,\n        )\n\n        # Stage 3 & 4: Multi-Candidate Synthesis + Test-Execution Self-Refinement + Regression Guard\n        guard = RegressionGuard(\n            repo_root=root,\n            allow_test_edits=self.allow_test_edits,\n            timeout_seconds=self.timeout_seconds,\n        )\n        all_evaluations: list[CandidateEvaluation] = []\n        seen_hashes: set[str] = set()\n        failure_feedback = traceback_text.strip()\n        rounds_executed = 0\n\n        for round_idx in range(self.max_refine_rounds + 1):\n            rounds_executed = round_idx + 1\n            proposals = self.runtime.synthesize_candidates(\n                repo_root=str(root),\n                issue_text=issue_text,\n                context_slice=ctx_slice,\n                failure_feedback=failure_feedback,\n                round_idx=round_idx,\n            )\n            if not proposals:\n                break\n\n            round_evals: list[CandidateEvaluation] = []\n            for cand_idx, prop in enumerate(proposals):\n                cand = CandidatePatch(\n                    candidate_id=f"r{round_idx}_c{cand_idx}_{prop.strategy}",\n                    strategy=prop.strategy,\n                    diff_text=prop.diff_text,\n                    temperature=prop.temperature,\n                    refine_round=round_idx,\n                    raw_thoughts=prop.raw_thoughts,\n                )\n                ev = guard.evaluate_candidate(\n                    candidate=cand,\n                    target_cmd=target_cmd,\n                    regression_cmd=regression_cmd,\n                    localization=localization,\n                )\n                if ev.canonical_hash not in seen_hashes:\n                    seen_hashes.add(ev.canonical_hash)\n                    all_evaluations.append(ev)\n                round_evals.append(ev)\n\n            # Check if a policy-compliant, zero-regression solution was found\n            passing = [\n                ev\n                for ev in round_evals\n                if ev.policy_compliant\n                and ev.apply_result.syntax_valid\n                and ev.target_report\n                and ev.target_report.passed\n                and ev.regression_report\n                and ev.regression_report.passed\n            ]\n            if passing:\n                break\n\n            # Prepare compact failure delta for next refinement round\n            best_round = (\n                DeterministicPatchRanker.rank(round_evals)[0] if round_evals else None\n            )\n            if best_round is not None:\n                failure_feedback = (\n                    best_round.rejection_reason or "No candidate passed tests"\n                )\n\n        # Stage 5: Deterministic Patch Ranking\n        ranked = DeterministicPatchRanker.rank(all_evaluations)\n        best = ranked[0] if ranked else None\n        solved = bool(\n            best is not None\n            and best.policy_compliant\n            and best.apply_result.syntax_valid\n            and best.target_report\n            and best.target_report.passed\n            and best.regression_report\n            and best.regression_report.passed\n        )\n        selected_patch = (\n            (best.apply_result.canonical_diff or best.candidate.diff_text)\n            if (best is not None and solved)\n            else ""\n        )\n\n        if apply_best and solved and selected_patch:\n            apply_unified_diff(\n                root,\n                selected_patch,\n                write_to_disk=True,\n                allow_test_edits=self.allow_test_edits,\n            )\n\n        return AgentRunResult(\n            solved=solved,\n            selected_patch=selected_patch,\n            selected_evaluation=best,\n            ranked_candidates=ranked,\n            localization=localization,\n            context_slice=ctx_slice,\n            rounds_executed=rounds_executed,\n            telemetry={\n                "entities_indexed": localization.total_entities_indexed,\n                "files_indexed": localization.total_files_indexed,\n                "slice_tokens": ctx_slice.estimated_tokens,\n                "candidates_evaluated": len(all_evaluations),\n                "top_localized_file": localization.ranked_files[0][0]\n                if localization.ranked_files\n                else None,\n            },\n        )\n'}

pkg_dir = PKG_ROOT / 'gemma4_swe_agent'
pkg_dir.mkdir(parents=True, exist_ok=True)
for rel_name, content in EMBEDDED_PKG_FILES.items():
    out_f = pkg_dir / rel_name
    out_f.write_text(content, encoding='utf-8')
print(f'Materialized {len(EMBEDDED_PKG_FILES)} gemma4_swe_agent modules to {pkg_dir}')

In [ ]:
import tempfile
import unittest
from gemma4_swe_agent import (
    ASTFaultLocalizer,
    AutonomousSWEAgent,
    DeterministicRuleRuntime,
    MinimalContextSlicer,
    search_replace_to_unified_diff,
)
from gemma4_swe_agent.gemma4_runtime import SynthesisProposal

# Local 5-Stage End-to-End Verification (runs on both CPU and Kaggle GPU)
with tempfile.TemporaryDirectory() as tmp:
    repo = Path(tmp)
    (repo / "app").mkdir()
    (repo / "tests").mkdir()
    (repo / "app" / "__init__.py").write_text("", encoding="utf-8")
    (repo / "app" / "quota.py").write_text(
        "def allow_request(used: int, limit: int) -> bool:\n"
        "    if used < 0 or limit < 0:\n"
        "        return False\n"
        "    if used < limit:\n"
        "        return True\n"
        "    return False\n",
        encoding="utf-8",
    )
    (repo / "tests" / "test_quota.py").write_text(
        "import unittest\n"
        "from app.quota import allow_request\n"
        "class TQuota(unittest.TestCase):\n"
        "    def test_exact_boundary(self):\n"
        "        self.assertTrue(allow_request(5, 5))\n",
        encoding="utf-8",
    )
    (repo / "tests" / "test_reg.py").write_text(
        "import unittest\n"
        "from app.quota import allow_request\n"
        "class TReg(unittest.TestCase):\n"
        "    def test_neg_and_over(self):\n"
        "        self.assertFalse(allow_request(-1, 5))\n"
        "        self.assertFalse(allow_request(6, 5))\n",
        encoding="utf-8",
    )

    cheat = search_replace_to_unified_diff(
        repo, [("tests/test_quota.py", "self.assertTrue(allow_request(5, 5))", "self.assertFalse(allow_request(5, 5))")]
    )
    regressive = search_replace_to_unified_diff(
        repo, [("app/quota.py", "    if used < 0 or limit < 0:\n        return False", "    return True")]
    )
    clean = search_replace_to_unified_diff(
        repo, [("app/quota.py", "    if used < limit:", "    if used <= limit:")]
    )
    runtime = DeterministicRuleRuntime({
        0: [SynthesisProposal("cheat_test", cheat, 0.7), SynthesisProposal("regressive", regressive, 0.35)],
        1: [SynthesisProposal("refined_bound", clean, 0.0)],
    })
    agent = AutonomousSWEAgent(runtime=runtime, max_refine_rounds=2)
    result = agent.solve_issue(
        repo_root=repo,
        issue_text="allow_request rejects exact boundary used == limit",
        target_cmd=[sys.executable, "-m", "unittest", "tests/test_quota.py"],
        regression_cmd=[sys.executable, "-m", "unittest", "tests/test_reg.py"],
        apply_best=True,
    )
    assert result.solved, "Expected self-refinement round to solve boundary bug"
    assert result.rounds_executed == 2
    assert result.selected_evaluation.candidate.strategy == "refined_bound"
    print("Benchmark telemetry:", json.dumps(result.telemetry, indent=2))
    print("Selected canonical patch:\n" + result.selected_patch)

In [ ]:
# Gemma 4 Post-Training (QLoRA SFT / Execution-Verified GRPO) & vLLM Serving Blueprint
# Toggle RUN_QLORA_TRAIN = True when Gemma 4 model weights are mounted in /kaggle/input on L4/H100 GPUs.
RUN_QLORA_TRAIN = False
GEMMA4_MODEL_PATH = "/kaggle/input/gemma-4/transformers/gemma-4-e4b-it/1"

QLORA_TRAINING_CONFIG = {
    "base_model_target": "gemma-4-31b-it-qat-w4a16-ct",
    "edge_model_target": "gemma-4-e4b-it-qat-w4a16-ct",
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    "reward_shaping": {
        "target_tests_pass": 1.0,
        "zero_regressions_bonus": 0.5,
        "minimal_blast_radius_max_bonus": 0.2,
        "syntax_error_penalty": -1.5,
        "forbidden_test_edit_penalty": -2.0,
    },
    "vllm_launch_flags": [
        "--reasoning-parser", "gemma4",
        "--enable-auto-tool-choice",
        "--tool-call-parser", "gemma4",
        "--max-model-len", "32768",
    ],
}
print("Gemma 4 Training & Serving Spec:", json.dumps(QLORA_TRAINING_CONFIG, indent=2))

if RUN_QLORA_TRAIN and Path(GEMMA4_MODEL_PATH).exists():
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from peft import LoraConfig, get_peft_model

    bnb_cfg = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    tokenizer = AutoTokenizer.from_pretrained(GEMMA4_MODEL_PATH)
    base_model = AutoModelForCausalLM.from_pretrained(
        GEMMA4_MODEL_PATH, quantization_config=bnb_cfg, device_map="auto"
    )
    lora_cfg = LoraConfig(
        r=QLORA_TRAINING_CONFIG["lora_r"],
        lora_alpha=QLORA_TRAINING_CONFIG["lora_alpha"],
        lora_dropout=QLORA_TRAINING_CONFIG["lora_dropout"],
        target_modules=QLORA_TRAINING_CONFIG["target_modules"],
        task_type="CAUSAL_LM",
    )
    peft_model = get_peft_model(base_model, lora_cfg)
    peft_model.print_trainable_parameters()

In [ ]:
EMBEDDED_BUNDLE_FILES = {'agent.yaml': 'name: gemma4_graphfix_swe\nagent_class: LlmAgent\nmodel: gemma-4-31b-it-qat-w4a16-ct\ndescription: >\n  Gemma4-GraphFix Autonomous SWE Developer Agent combining AST Call-Graph Fault\n  Localization, Minimal Context Slicing, Multi-Candidate Patch Synthesis,\n  Regression Guard, and Deterministic Patch Ranking.\ninstruction: !include prompts/system.md\ntools:\n  - run_command\n  - read_file\n  - edit_file\n  - write_file\n  - get_status\n  - submit_patch\n  - get_code_neighbors\n  - search_similar_code\n  - get_code_subgraph\ngenerate_content_config: !include configs/sampling.yaml\n', 'agent_e4b_fast.yaml': 'name: gemma4_graphfix_swe_e4b\nagent_class: LlmAgent\nmodel: gemma-4-e4b-it-qat-w4a16-ct\ndescription: >\n  Lightweight E4B QAT configuration of Gemma4-GraphFix optimized for high\n  test-time compute throughput and Edge/Hardware efficiency track eligibility.\ninstruction: !include prompts/system.md\ntools:\n  - run_command\n  - read_file\n  - edit_file\n  - write_file\n  - get_status\n  - submit_patch\n  - get_code_neighbors\n  - search_similar_code\n  - get_code_subgraph\ngenerate_content_config: !include configs/sampling.yaml\n', 'configs/sampling.yaml': 'temperature: 0.2\ntop_p: 0.95\ntop_k: 64\nmax_output_tokens: 4096\nthinking_config:\n  thinking_level: MEDIUM\n', 'eval_config.yaml': 'max_time_minutes: 8\nmax_tool_calls: 36\ntimeout_seconds: 180\nmax_turns: 60\n', 'prompts/self_refine.md': '# Self-Refinement Delta Protocol\n\nWhen a candidate patch fails verification:\n1. Classify the failure: `SYNTAX_ERROR`, `ASSERTION_MISMATCH`, `REGRESSION_BROKEN`, `EDGE_CASE_EXCEPTION`, or `TIMEOUT`.\n2. Inspect only the innermost traceback frame and compare expected vs actual values.\n3. If `REGRESSION_BROKEN`, inspect `git diff` via `get_status` and tighten the conditional guard instead of broadening behavior.\n4. Keep total blast radius minimal (minimum files, hunks, and added/removed lines).\n', 'prompts/system.md': '# Gemma4-GraphFix Autonomous SWE Agent Protocol\n\nYou are an autonomous software engineering agent operating inside an isolated `/workspace` repository under a strict tool budget (36 tool calls, 8 minutes max per issue). Follow the 5-stage engineering loop strictly:\n\n## Stage 1 — AST Fault Localization (Turns 1–4)\n1. Identify target symbols, filenames, and stack frames from the issue description.\n2. Call `get_code_subgraph` and `get_code_neighbors` around the top suspect symbols, or run `python skills/ast_swe_toolkit/scripts/run_localize.py --repo . --issue "<summary>"` via `run_command` to obtain ranked AST entities and linked test files.\n3. Separate production source files from test files (`tests/`, `test_*.py`).\n\n## Stage 2 — Minimal Context Slicing (Turns 5–7)\n1. Read ONLY the localized line spans (`read_file` with targeted ranges) plus module imports and 1-hop caller/callee signatures. Never dump whole large files into context.\n2. Run the failing test once via `run_command` to observe the exact failing assertion and baseline state before editing.\n\n## Stage 3 — Multi-Candidate Patch Synthesis (Turns 8–12)\n1. Formulate up to 3 minimal repair hypotheses (root-cause fix, boundary/null guard, state invariant) before modifying code.\n2. Prefer the smallest surgical edit (`edit_file`) touching the fewest lines and hunks.\n3. **HARD RULE (Anti-Cheat / Regression Guard):** NEVER edit, skip, weaken, or delete existing test files (`tests/`, `test_*.py`, `*_test.py`) unless the issue explicitly requires updating a broken test expectation.\n\n## Stage 4 — Test-Execution Self-Refinement & Regression Guard (Turns 13–24)\n1. After each `edit_file`, execute BOTH the failing target tests AND related module regression tests (`pytest -q`).\n2. If tests fail, parse only the traceback delta (exception type, failing line, diff vs expected) and refine cleanly—never stack redundant workarounds. Use `get_status` to verify clean minimal diff scope.\n3. Verify zero syntax errors (`python -m py_compile <edited_files>`) before finalizing.\n\n## Stage 5 — Deterministic Patch Ranking & Submission (Final Turn)\n1. Confirm: (a) target tests PASS, (b) baseline regression tests PASS, (c) `get_status` shows only minimal production code changes.\n2. Call `submit_patch` immediately once verified. Do not waste remaining tool calls on speculative refactors.\n', 'skills/ast_swe_toolkit/scripts/run_localize.py': '"""Self-contained AST fault localizer + minimal slicer CLI bundled inside ADK skills/."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport ast\nimport math\nimport os\nimport re\nfrom collections import defaultdict\nfrom pathlib import Path\n\nIGNORED_DIRS = {".git", "__pycache__", ".pytest_cache", ".venv", "venv", "build", "dist"}\n\n\ndef is_test_path(rel_path: str) -> bool:\n    norm = rel_path.replace("\\\\", "/").lower()\n    parts = norm.split("/")\n    if any(p in ("tests", "test", "testing") for p in parts[:-1]):\n        return True\n    name = parts[-1]\n    return name.startswith("test_") or name.endswith("_test.py")\n\n\ndef tokenize(text: str) -> set[str]:\n    out: set[str] = set()\n    for word in re.findall(r"[A-Za-z_][A-Za-z0-9_]*", text):\n        for part in word.split("_"):\n            for cp in re.findall(r"[A-Z]+(?=[A-Z][a-z0-9]|$)|[A-Z]?[a-z]+|[0-9]+", part):\n                low = cp.lower()\n                if len(low) >= 2:\n                    out.add(low)\n    return out\n\n\ndef localize_repo(repo_root: Path, query: str, top_k: int = 5) -> str:\n    q_tokens = tokenize(query)\n    exact = set(re.findall(r"\\b[A-Za-z_][A-Za-z0-9_]{2,}\\b", query))\n    candidates: list[tuple[float, str, str, int, int]] = []\n    tests: list[str] = []\n\n    for root, dirs, files in os.walk(repo_root):\n        dirs[:] = sorted(d for d in dirs if d not in IGNORED_DIRS and not d.startswith("."))\n        for fname in sorted(files):\n            if not fname.endswith(".py"):\n                continue\n            fpath = Path(root) / fname\n            rel = fpath.relative_to(repo_root).as_posix()\n            text = fpath.read_text(encoding="utf-8", errors="replace")\n            try:\n                tree = ast.parse(text)\n            except SyntaxError:\n                continue\n            lines = text.splitlines()\n            if is_test_path(rel):\n                if tokenize(text) & q_tokens:\n                    tests.append(rel)\n                continue\n            for node in ast.walk(tree):\n                if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):\n                    end_ln = getattr(node, "end_lineno", node.lineno) or node.lineno\n                    body = "\\n".join(lines[node.lineno - 1 : end_ln])\n                    overlap = tokenize(body) & q_tokens\n                    score = len(overlap) / (1.0 + 0.1 * math.log1p(max(1, end_ln - node.lineno + 1)))\n                    if node.name in exact:\n                        score += 3.0\n                    if rel in query:\n                        score += 2.0\n                    if score > 0:\n                        candidates.append((round(score, 3), rel, node.name, node.lineno, end_ln))\n\n    candidates.sort(key=lambda x: (-x[0], x[1], x[3]))\n    out = ["[AST_LOCALIZER_RESULTS]"]\n    for score, rel, name, s_ln, e_ln in candidates[:top_k]:\n        out.append(f"- {rel}:{s_ln}-{e_ln} `{name}` (score={score})")\n        lines = (repo_root / rel).read_text(encoding="utf-8", errors="replace").splitlines()\n        for ln in range(s_ln, min(len(lines), e_ln) + 1):\n            out.append(f"  L{ln:04d}: {lines[ln - 1]}")\n    if tests:\n        out.append("[LINKED_TESTS_DO_NOT_EDIT]")\n        out.extend(f"- {t}" for t in tests[:5])\n    return "\\n".join(out)\n\n\ndef main() -> None:\n    ap = argparse.ArgumentParser(description="Run local AST fault localization inside sandbox")\n    ap.add_argument("--repo", type=Path, default=Path("."))\n    ap.add_argument("--issue", required=True)\n    ap.add_argument("--top-k", type=int, default=4)\n    args = ap.parse_args()\n    print(localize_repo(args.repo.resolve(), args.issue, top_k=args.top_k))\n\n\nif __name__ == "__main__":\n    main()\n', 'skills/ast_swe_toolkit/SKILL.md': '---\nname: ast_swe_toolkit\ndescription: Static AST Fault Localization, Minimal Context Slicing, and Patch Syntax/Guard validation for Python repositories inside the Kaggle sandbox.\n---\n\n# AST SWE Toolkit Skill\n\nUse `scripts/run_localize.py` via `run_command` before making edits on unfamiliar or multi-file Python repositories:\n\n```bash\npython skills/ast_swe_toolkit/scripts/run_localize.py --repo . --issue "<issue summary>" --max-tokens 2600\n```\n\nIt returns:\n1. Top suspicious production entities ranked by AST call-graph + traceback + BM25 IDF overlap.\n2. Line-numbered minimal context slice (`L0001: ...`) ready for exact `edit_file` calls.\n3. Linked read-only test files to execute after patching.\n'}

import ast
import zipfile
import yaml

bundle_out = WORK_ROOT / "submission_bundle"
for rel_path, content in EMBEDDED_BUNDLE_FILES.items():
    dst = bundle_out / rel_path
    dst.parent.mkdir(parents=True, exist_ok=True)
    dst.write_text(content, encoding="utf-8")

# Validate root agent.yaml, !include targets, and sampling config (no deprecated thinking_budget)
agent_text = (bundle_out / "agent.yaml").read_text(encoding="utf-8")
sampling_cfg = yaml.safe_load((bundle_out / "configs" / "sampling.yaml").read_text(encoding="utf-8"))
assert "thinking_budget" not in sampling_cfg and "thinking_budget" not in (sampling_cfg.get("thinking_config") or {})
for rel_path, content in EMBEDDED_BUNDLE_FILES.items():
    if rel_path.endswith(".py"):
        ast.parse(content, filename=rel_path)

sub_zip = WORK_ROOT / "submission.zip"
with zipfile.ZipFile(sub_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for rel_path in sorted(EMBEDDED_BUNDLE_FILES):
        zf.write(bundle_out / rel_path, arcname=rel_path)

with zipfile.ZipFile(sub_zip, "r") as zf:
    names = zf.namelist()
    assert "agent.yaml" in names, "Missing root agent.yaml in submission.zip"

print(f"SUCCESS: Built {sub_zip} ({sub_zip.stat().st_size} bytes), files={len(names)}")
print("Root agent.yaml validated; sampling config:", sampling_cfg)